In [1]:
from crewai import Agent, Task, LLM, Crew
from pydantic import BaseModel, Field
from typing import List
import os
from crewai.flow.flow import Flow, start, listen, and_

In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
llm = LLM(model="gpt-4o")

In [3]:
# Agents

curate_learning_agent = Agent(
    llm=llm,
    role="Learning Curator",
    backstory="You are an expert in curating personalized learning experiences for students.",
    goal="To provide tailored educational content that meets the unique needs of each student.",
    verbose=True
)

create_quizes_agent = Agent(
    llm=llm,
    role="Quiz Creator",
    backstory="You are an expert in creating engaging and effective quizzes for students.",
    goal="To design quizzes that accurately assess student understanding and promote learning.",
    verbose=True
)

project_recommendation_agent = Agent(
    llm=llm,
    role="Project Recommendation Specialist",
    backstory="You are an expert in recommending educational projects that align with students' interests, learning goals and align with their expertise level (rated out of 5).",
    goal="To suggest projects that enhance students' learning experiences and foster creativity.",
    verbose=True
)   

summary_agent = Agent(
    llm=llm,
    role="Summary Agent",
    backstory="You are an expert in summarizing educational content and quizzes.",
    goal="To provide concise summaries of the curated learning content, quizzes and recommended projects for easy reference. Also in case anything is missing in the content, quizzes or projects, inform the user about it in the beginning of the summary.",
    verbose=True
)


In [4]:
# Tasks

def content_selection_task(topic_of_interest:str)->Task:
    """
    Task to curate personalized learning content based on the student's topic of interest.
    """
    return Task(
        description=f"Curate personalized learning content based on the student's topic of interest : {topic_of_interest}",
        agent=curate_learning_agent,
        expected_output="A curated list of personalized learning content that aligns with the student's topic of interest."
        )

def quiz_creation_task(content_generated)->Task:
    """
    Task to create quizzes based on the curated learning content.
    """
    return Task(
        description=f"Create quizzes based on the curated learning content: {content_generated}",
        agent=create_quizes_agent,
        expected_output="A set of quizzes that effectively assess student understanding of the curated content."
        )

def project_recommendation_task(content_generated, expertise_lvl:int)->Task:
    """
    Task to recommend educational projects based on the curated learning content and student's expertise level.
    """
    return Task(
        description=f"Recommend educational projects based on the curated learning content: {content_generated} and student's expertise level (rating out of 5): {expertise_lvl}",
        agent=project_recommendation_agent,
        expected_output="A list of recommended educational projects that align with the curated content and the student's expertise level."
        )

def summary_task(content_generated, quizes_generated, projects_recommended)->Task:
    """
    Task to summarize the curated learning content, quizzes, and recommended projects.
    """
    return Task(
        description=f"Summarize the curated learning content: {content_generated}, quizzes: {quizes_generated}, and recommended projects: {projects_recommended}",
        agent=summary_agent,
        expected_output="A concise summary of the curated learning content, quizzes, and recommended projects for easy reference. In case anything is missing in the content, quizzes or projects, inform the user about it in the beginning of the summary."
        )


In [14]:
# Flow
class EducationAssistant(BaseModel):
    content_generated: str = Field(default="", description="The curated learning content generated for the student.")
    quizes_generated: str = Field(default="", description="The quizzes generated based on the curated learning content.")
    projects_recommended: str = Field(default="", description="The educational projects recommended based on the curated learning content and student's expertise level.")

In [17]:
class PersonalisedEducationFlow(Flow[EducationAssistant]):
    """
    Flow to curate personalized learning content, create quizzes, and recommend educational projects for students.
    """
    topic_of_interest: str = Field(..., description="The topic of interest for the student.")
    expertise_lvl: int = Field(..., description="The expertise level of the student (rated out of 5).")

    def __init__(self, topic_of_interest: str, expertise_lvl: int):
        super().__init__(topic_of_interest=topic_of_interest, expertise_lvl=expertise_lvl)

    @start()
    def curate_content(self):
        """
        Start the flow by curating personalized learning content based on the student's topic of interest.
        """
        print("Curating personalized learning content...")
        task = content_selection_task(self.topic_of_interest)
        crew = Crew(agents=[curate_learning_agent], tasks=[task])
        self.state.content_generated = crew.kickoff()
        print(f"Content Curated")

    @listen(curate_content)
    def create_quizzes(self):
        """
        Create quizzes based on the curated learning content.
        """
        print("Creating quizzes based on the curated learning content...")
        task = quiz_creation_task(self.state.content_generated)
        crew = Crew(agents=[create_quizes_agent], tasks=[task])
        self.state.quizes_generated = crew.kickoff()
        print(f"Quizzes Created")

    @listen(curate_content)
    def recommend_projects(self):
        """
        Recommend educational projects based on the curated learning content and student's expertise level.
        """
        print("Recommending educational projects...")
        task = project_recommendation_task(self.state.content_generated, self.expertise_lvl)
        crew = Crew(agents=[project_recommendation_agent], tasks=[task])
        self.state.projects_recommended = crew.kickoff()
        print(f"Projects Recommended")

    @listen(and_(create_quizzes, recommend_projects))
    def summarize_results(self):
        """
        Summarize the results of the flow, including curated content, quizzes, and recommended projects.
        """
        print("Summarizing results...")
        task = summary_task(self.state.content_generated, self.state.quizes_generated, self.state.projects_recommended)
        crew = Crew(agents=[summary_agent], tasks=[task])
        summary = crew.kickoff()
        print(f"Summary: {summary}")

In [18]:
import nest_asyncio
nest_asyncio.apply()

def run_personalized_education_flow(topic_of_interest: str, expertise_lvl: int):
    """
    Function to run the personalized education flow.
    """
    flow = PersonalisedEducationFlow(topic_of_interest=topic_of_interest, expertise_lvl=expertise_lvl)
    flow.kickoff()

run_personalized_education_flow("Machine Learning", 3)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PersonalisedEducationFlow                                                                                │
│  ID: 7ff447eb-9d9b-43f6-8948-233f22baf873                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PersonalisedEducationFlow                                                                                │
│  ID: 7ff447eb-9d9b-43f6-8948-233f22baf873                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 7ff447eb-9d9b-43f6-8948-233f22baf873

Curating personalized learning content...


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: curate_content                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Agent                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The curated learning content provides a comprehensive overview to guide learners from basic to advanced        │
│  machine learning concepts through diverse formats including online courses, books, and articles alongside      │
│  practical projects.                                                                                            │
│                                                                                                                 │
│  ### Online Courses and Tutorials:                                                                              │
│  - **Coursera - Machine Learning** by Stanford: Foundational concepts, supervised and unsupervised learning.    │
│  - **edX - Principles of Machine Learning**: Basics of data-driven learning.                                    │
│  - **Udacity - Intro to Machine Learning with PyTorch and TensorFlow**: Practical framework guidance.           │
│                                                                                                                 │
│  ### Books:                                                                                                     │
│  - **"Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow"**: Python-based, hands-on             │
│  techniques.                                                                                                    │
│  - **"Pattern Recognition and Machine Learning"**: Comprehensive introduction.                                  │
│  - **"Deep Learning"**: In-depth exploration by experts.                                                        │
│                                                                                                                 │
│  ### Articles and Papers:                                                                                       │
│  - **"A Few Useful Things to Know About Machine Learning"**: Practical insights and pitfalls.                   │
│  - **"Understanding Machine Learning: From Theory to Algorithms"**: Covers theoretical and practical aspects.   │
│                                                                                                                 │
│  ### Practical Implementation:                                                                                  │
│  - **Kaggle**: Real-world competitions and datasets for practice.                                               │
│  - **Google's Machine Learning Crash Course**: Practical-focused learning with TensorFlow.                      │
│                                                                                                                 │
│  ### Additional Resources:                                                                                      │
│  - **Podcasts**: "The TWIML AI Podcast"                                                                         │
│  - **YouTube Channels**: "Sentdex" for Python-based ML                                                          │
│                                                                                                                 │
│  ### Quizzes:                                                                                                   │
│  Quizzes aim to evaluate understanding and application skills across various levels from courses, books, and    │
│  implementation experiences, focusing on different mach

Summary: The curated learning content provides a comprehensive overview to guide learners from basic to advanced machine learning concepts through diverse formats including online courses, books, and articles alongside practical projects.

### Online Courses and Tutorials:
- **Coursera - Machine Learning** by Stanford: Foundational concepts, supervised and unsupervised learning.
- **edX - Principles of Machine Learning**: Basics of data-driven learning.
- **Udacity - Intro to Machine Learning with PyTorch and TensorFlow**: Practical framework guidance.

### Books:
- **"Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow"**: Python-based, hands-on techniques.
- **"Pattern Recognition and Machine Learning"**: Comprehensive introduction.
- **"Deep Learning"**: In-depth exploration by experts.

### Articles and Papers:
- **"A Few Useful Things to Know About Machine Learning"**: Practical insights and pitfalls.
- **"Understanding Machine Learning: From Theory to Algorithms"**

/Users/varun.tyagi/Documents/learning workspace/analytics_vidhya/code_agentic_ai_pioneer/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content_generated', input_value=CrewOutput(raw='To provid... successful_requests=1)), input_type=CrewOutput])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='quizes_generated', input_value=CrewOutput(raw='Below is ... successful_requests=2)), input_type=CrewOutput])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='projects_recommended', input_value=CrewOutput(raw='Based on ... successful_requests=3)), input_type=CrewOutput])
  return self.__pydantic_serializer__.to_python(


╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PersonalisedEducationFlow                                                                                │
│  ID: 7ff447eb-9d9b-43f6-8948-233f22baf873                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯